In [1]:
import numpy as np
from scipy.signal import cont2discrete

from config.matrizes_config import matrizes
from S_MPC import carnaval
from Yr_MPC import alvo
from Y0_MPC import y_0
import casadi as cs

In [2]:
Ts = 1.0               # tempo de amostragem
N = 12                 # horizonte do modelo
P = 17                 # horizonte de predição
M = 5                  # horizonte de controle
m = 2
r = 2

In [3]:
S = carnaval(m, r, M, P, N, 1.0)
S

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.25935969,  0.01922138,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.64839921,  0.24725846,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.44304349,  0.06162445, -0.25935969,  0.01922138,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 1.10760874,  0.34935357,  0.64839921,  0.24725846,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.57313211,  0.11194321, -0.44304349,  0.06162445, -0.25935969,
         0.01922138,  0.        ,  0.        

In [4]:
y_k = np.array([3.4, 1.8])
alfa = np.array([0.8, 0.3])
ysp = np.array([0.34, 0.23])
YR = alvo(alfa, y_k, y_sp = ysp, m = m, P = P)

In [5]:
u_final = [3, 7]
len_u = r*N-2
delta_u = np.zeros(len_u)

for i in range(len_u):
    delta_u[i] = np.random.rand()

Y0 = y_0(m, P, N, S, delta_u, u_final)

In [6]:
E0 = YR - Y0 
print(S.shape)  # mP x rM
print(YR.shape)    # mP
print(Y0.shape)    # mP
print(E0.shape)    # mP

(34, 10)
(34,)
(34,)
(34,)


In [8]:
# pesos das saídas
q_X = 1.0
q_S = 1.0

# pesos das entradas
r_D = 0.1
r_Sf = 0.1

Q = np.diag([q_X, q_S])      # 2x2
R = np.diag([r_D, r_Sf])     # 2x2

Qbar = np.kron(np.eye(P), Q) # 2P x 2P
Rbar = np.kron(np.eye(M), R) # 2M x 2M

print(Qbar)
print(Rbar)

[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]
[[0.1 0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.1 0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.1 0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.1 0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.1 0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.1 0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.1 0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.1 0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.1 0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.1]]
